# Discover subprocess tools and attach them to an agent: practice

Describe command/args → open stdio server connection → list tools → keep servers alive while agent runs → close resources.

Use this with [the walkthrough](walkthrough.md).

**Source:**

- [6_mcp/1_lab1.ipynb](../../../../../6_mcp/1_lab1.ipynb)

**Run one cell at a time:** predict → run with Shift+Enter → explain → change one thing. Select the root project .venv kernel.

**Offline adaptation:** code below uses fictional data or local libraries. It never executes the linked source, reads a credential file, sends messages, starts a UI/server, or changes the course databases. Simulations are labeled; they do not verify external integrations.

## 1. MCP tools, resources, and server lifetime

A tool performs an operation; a resource supplies readable context. A client discovers tool descriptions and calls them while the server is connected. This fixture is not a protocol transport.

**Predict:** Is listing tool names the same as executing them?

In [ ]:
from contextlib import contextmanager
calls = []
@contextmanager
def practice_server():
    tools = {"add": lambda a, b: a + b}
    try:
        calls.append("connected")
        yield tools
    finally:
        calls.append("closed")
with practice_server() as tools:
    print("Discovered:", list(tools))
    result = tools["add"](12, 30)
    print("Called add:", result)
assert result == 42 and calls == ["connected", "closed"]

**Check:** No. Discovery returns descriptions. A real stdio subprocess also has a working directory, environment, and protocol stream.

**Change one thing:** Add a read-only dictionary resource and a tool returning a fixed date. Explain how discovery differs from invocation.

## 2. A tool schema describes; a dispatcher executes

The model can request a function call. Your application decodes arguments, recognizes the name, executes the function, and returns a matching tool result. No model is needed to practice this boundary.

**Predict:** Does defining the schema call the function?

In [ ]:
import json
records = []
def record_interest(email):
    records.append(email)  # In memory only; does not send email or notifications.
    return {"recorded": True}
schema = {
    "type": "function", "function": {
        "name": "record_interest", "description": "Record a supplied address",
        "parameters": {"type": "object", "properties": {"email": {"type": "string"}},
                       "required": ["email"], "additionalProperties": False},
    },
}
assert records == []
requested = {"id": "call_1", "name": "record_interest", "arguments": '{"email":"learner@example.com"}'}
allowed = {"record_interest": record_interest}  # An explicit registry of permitted functions.
if requested["name"] not in allowed:
    raise ValueError("Unknown tool")
result = allowed[requested["name"]](**json.loads(requested["arguments"]))
tool_message = {"role": "tool", "tool_call_id": requested["id"], "content": json.dumps(result)}
print(tool_message)
assert records == ["learner@example.com"]

**Check:** No. Only the explicit dispatcher call changes records. Keep the assistant tool-call message before the matching tool result in a full conversation.

**Change one thing:** Change the requested name to an unknown function and observe the error. Then add a harmless second tool to the registry.

## 3. Browser tools have setup, action, and cleanup

A browser toolkit makes actions such as navigation and extraction callable. The practice object imitates that lifetime with fixed HTML; it launches no browser and accesses no website.

**Predict:** Can extraction happen before navigation in this demonstration?

In [ ]:
class PracticeBrowser:
    def __init__(self):
        self.page = None
        self.closed = False
    def navigate(self, html):
        self.page = html
    def extract(self):
        if self.page is None:
            raise RuntimeError("Navigate first")
        return self.page.replace("<p>", "").replace("</p>", "")
    def close(self):
        self.closed = True
browser = PracticeBrowser()
try:
    browser.navigate("<p>Fictional page text</p>")
    print(browser.extract())
finally:
    browser.close()
assert browser.closed

**Check:** No in this fixture. A fake browser teaches sequence only; it does not validate Playwright or real page behavior.

**Change one thing:** Call extract before navigate and inspect the error. Explain why real Playwright also needs binaries and cleanup.

## 4. Configuration stays outside executable examples

Environment-variable names identify settings. Values belong in .environment/.env and are loaded deliberately. This exercise inspects only the path and simulated presence flags; it reads no private file or variable values.

**Predict:** Does finding a file prove that authentication works?

In [ ]:
from pathlib import Path
# Find the repository from a kernel started inside it.
root = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "1_foundations" / "3_lab3.ipynb").is_file()), None)
if root is None:
    raise FileNotFoundError("Start your kernel inside ai_agents")
configuration_path = root / ".environment" / ".env"
print("Configuration path:", configuration_path)
simulated_presence = {"OPENAI_API_KEY": True, "GOOGLE_API_KEY": False}
print(simulated_presence)  # Booleans only; these are fictional flags.
assert configuration_path.name == ".env"

**Check:** No. File existence, variable presence, and successful provider authentication are separate checks.

**Change one thing:** Explain the difference between a missing file, a missing variable, and a provider rejecting authentication. Do not print token prefixes or values.

## Rebuild from memory

1. Recreate the main data-flow step in a new scratch cell.
2. Add one case that should fail and identify its exception/stop condition.
3. Compare your version with the linked source cells or functions.
4. Explain where a real model or service would enter, and what it could change.

For deliberate original-source execution, see the shared [configuration guide](../../../configuration.md). All provider/transport integrations remain UNVERIFIED at runtime.